In [ ]:
import noisereduce as nr
from pydub import AudioSegment
import librosa
import soundfile as sf
import numpy as np
import pydub
from pydub.effects import normalize

In [ ]:


# Шаг 1: Загрузка и проверка аудио
audio, sample_rate = sf.read("src/Боевая-пехотная.wav")

# Проверка формы аудио
print(f"Audio shape: {audio.shape}, Sample rate: {sample_rate}")

# Преобразование в моно, если стерео
if len(audio.shape) > 1 and audio.shape[1] > 1:
    audio = np.mean(audio, axis=1)  # Среднее по каналам

# Шаг 2: Подавление шумов
# Обрабатываем аудио по частям, чтобы избежать перегрузки памяти
chunk_size = 10 * sample_rate  # 10 секунд
clean_audio = []

for i in range(0, len(audio), chunk_size):
    chunk = audio[i:i + chunk_size]
    clean_chunk = nr.reduce_noise(y=chunk, sr=sample_rate, prop_decrease=0.8)
    clean_audio.append(clean_chunk)

clean_audio = np.concatenate(clean_audio)

# Сохранение очищенного аудио
sf.write("clean_song.wav", clean_audio, sample_rate)

# Шаг 3: Улучшение вокала и подавление громкой музыки
audio = AudioSegment.from_file("clean_song.wav")
audio = normalize(audio)
audio = audio.high_pass_filter(100)  # Убрать басы
audio = audio.low_pass_filter(10000)  # Убрать высокие частоты
audio = audio + 4  # Усилить вокал
audio = audio.compress_dynamic_range(threshold=-20.0, ratio=5.0)
audio.export("enhanced_song.wav", format="wav")


In [ ]:
# Параметры
input_file = "enhanced_song.wav"
output_file = "g.wav"
sample_rate = 44100  # Частота дискретизации
chunk_duration = 10  # Длительность чанка (секунды)

# Шаг 1: Загрузка и обработка аудио по чанкам
audio, sr = librosa.load(input_file, sr=sample_rate, mono=True)
chunk_size = chunk_duration * sample_rate
clean_audio = []

for i in range(0, len(audio), chunk_size):
    chunk = audio[i:i + chunk_size]
    # Подавление шумов (pre-emphasis фильтр)
    clean_chunk = librosa.effects.preemphasis(chunk, coef=0.97)
    clean_audio.append(clean_chunk)

clean_audio = np.concatenate(clean_audio)
sf.write("clean_song.wav", clean_audio, sample_rate)

# Шаг 2: Усиление вокала и подавление громкой музыки
audio_segment = pydub.AudioSegment.from_file("clean_song.wav")

# Нормализация громкости
audio_segment = normalize(audio_segment)

# Эквализация: усиление вокала (1–4 кГц), подавление басов и высоких частот
audio_segment = audio_segment.high_pass_filter(100)  # Убрать басы
audio_segment = audio_segment.low_pass_filter(10000)  # Убрать высокие частоты
audio_segment = audio_segment + 6  # Усилить вокал на 6 дБ

# Компрессия для выравнивания громкости (подавление громкой музыки)
audio_segment = audio_segment.compress_dynamic_range(threshold=-20.0, ratio=6.0)

# Сохранение результата
audio_segment.export(output_file, format="wav")

print(f"Обработка завершена. Результат сохранен в {output_file}")
